# Farm Yield Prediction - Yield Model Training

This notebook trains the **CatBoost Regressor** for yield prediction (kg/ha).
Optimized for Google Colab with T4 GPU.

In [ ]:
!pip install catboost pandas numpy scikit-learn

In [ ]:
import pandas as pd
import os
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from google.colab import files

### Upload your dataset
Upload `nigeria_agri_yield_enhanced.csv` from your local `data/` folder.

In [ ]:
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print(f"Using dataset: {DATA_PATH}")

In [ ]:
# Load and prepare data
df = pd.read_csv(DATA_PATH)

# Feature engineering
df['soil_fertility_index'] = df['soil_nitrogen'] + df['soil_phosphorus'] + df['soil_potassium']
df['np_ratio'] = df['soil_nitrogen'] / (df['soil_phosphorus'] + 1e-6)
df['climate_index'] = df['rainfall_mm'] * df['temperature_C']

# Drop rows where target is missing
df = df.dropna(subset=['yield_kg_ha'])

target_col = 'yield_kg_ha'
X = df.drop(columns=[target_col, 'region', 'state', 'pest_type', 'pest_severity', 'labor_input', 'farm_size_ha', 'rainfall_variability', 'extreme_weather', 'temperature_stress', 'soil_degradation'], errors='ignore')
y = df[target_col]

# Identify column types
cat_features = X.select_dtypes(include=['object']).columns.tolist()
num_features = X.select_dtypes(include=['number']).columns.tolist()

# Fill NaN in categorical columns (CatBoost crashes on NaN in cat features)
for col in cat_features:
    X[col] = X[col].fillna('Unknown')

# Fill NaN in numeric columns
for col in num_features:
    X[col] = X[col].fillna(X[col].median())

print(f"Dataset: {len(X)} rows")
print(f"Features: {X.shape[1]} ({len(cat_features)} categorical, {len(num_features)} numeric)")
print(f"Target stats: mean={y.mean():.0f}, std={y.std():.0f}")
print(f"Remaining NaN: {X.isna().sum().sum()}")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train yield model
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    cat_features=cat_features,
    task_type='GPU' if os.environ.get('COLAB_GPU', '0') == '1' else 'CPU',
    verbose=200,
    early_stopping_rounds=50
)

model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

# Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n=== Yield Model Results ===")
print(f"RMSE: {rmse:.2f} kg/ha")
print(f"MAE:  {mae:.2f} kg/ha")
print(f"R2:   {r2:.4f}")

In [ ]:
# Save and download
model.save_model('catboost_model.cbm')
files.download('catboost_model.cbm')
print("Download started! Place this in your local models/ directory.")